# 03 — Salary Quantile Regression: Training & Evaluation

This notebook trains a `SalaryQuantileNet` model and evaluates it with:
1. **Calibration plots** — actual fraction below predicted quantile vs. nominal τ
2. **Median (q50) MAE** — mean absolute error of the median prediction
3. **Residual analysis** — distribution and patterns of prediction errors

> **Note:** Until Phase 2 (Embedding & Retrieval) is complete, this notebook uses **synthetic data** that mimics the expected input distribution. Swap in real `job_embeddings.npy` and `salaries.npy` once available.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is importable
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from ml.salary_model import (
    SalaryQuantileNet,
    PinballLoss,
    SalaryDataset,
    split_data,
    predict_salary,
    QUANTILES,
    SEED,
)

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print(f"PyTorch {torch.__version__}")
print(f"Quantiles: {QUANTILES}")

## 1. Data: Synthetic or Real

Toggle `USE_REAL_DATA = True` and set paths once Phase 2 outputs are available.

In [ ]:
USE_REAL_DATA = False

EMB_DIM = 384   # all-MiniLM-L6-v2
N_SAMPLES = 5000

if USE_REAL_DATA:
    embeddings = np.load(PROJECT_ROOT / "models" / "job_embeddings.npy")
    salaries   = np.load(PROJECT_ROOT / "data" / "processed" / "salaries.npy")
    EMB_DIM    = embeddings.shape[1]
    N_SAMPLES  = len(salaries)
    print(f"Loaded real data: {N_SAMPLES} samples, dim={EMB_DIM}")
else:
    # --- Synthetic data with learnable structure ---
    # Create embeddings that have a correlation with salary so the model
    # has something meaningful to learn.
    rng = np.random.default_rng(SEED)
    
    # Latent "seniority" signal embedded in first 10 dims
    seniority = rng.standard_normal(N_SAMPLES).astype(np.float32)
    noise_dims = rng.standard_normal((N_SAMPLES, EMB_DIM)).astype(np.float32) * 0.3
    for d in range(10):
        noise_dims[:, d] += seniority * (0.5 + 0.1 * d)
    
    # Normalise to unit vectors (mimicking SBERT output)
    norms = np.linalg.norm(noise_dims, axis=1, keepdims=True)
    embeddings = (noise_dims / norms).astype(np.float32)
    
    # Salary: base + seniority signal + noise  (realistic US tech range)
    salaries = (80_000 + seniority * 25_000 + rng.standard_normal(N_SAMPLES) * 15_000).astype(np.float32)
    salaries = np.clip(salaries, 30_000, 300_000)
    
    print(f"Synthetic data: {N_SAMPLES} samples, dim={EMB_DIM}")
    print(f"Salary range: ${salaries.min():,.0f} – ${salaries.max():,.0f}")
    print(f"Salary mean:  ${salaries.mean():,.0f}  median: ${np.median(salaries):,.0f}")

## 2. Train / Val / Test Split

In [ ]:
train_ds, val_ds, test_ds = split_data(embeddings, salaries, seed=SEED)
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")

BATCH_SIZE = 256
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

## 3. Model Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = SalaryQuantileNet(
    embedding_dim=EMB_DIM,
    n_extra_features=0,
    dropout=0.2,
).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {param_count:,}")

criterion = PinballLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5
)

# --- Training loop ---
EPOCHS = 150
PATIENCE = 15

best_val_loss = float("inf")
epochs_no_improve = 0
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    epoch_train = []
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        epoch_train.append(loss.item())
    
    # Validate
    model.eval()
    epoch_val = []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            epoch_val.append(criterion(model(X_b), y_b).item())
    
    avg_train = np.mean(epoch_train)
    avg_val = np.mean(epoch_val)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    scheduler.step(avg_val)
    
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        epochs_no_improve = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
    
    if epoch % 10 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:>3d}  train={avg_train:.2f}  val={avg_val:.2f}  lr={lr:.1e}")

# Restore best weights
model.load_state_dict(best_state)
model.eval()
print(f"\nBest val loss: {best_val_loss:.4f}  (stopped at epoch {len(train_losses)})")

### 3.1 Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train", linewidth=1.5)
ax.plot(val_losses, label="Validation", linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Pinball Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 4. Test-Set Evaluation

Collect predictions on the held-out test set.

In [ ]:
model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        preds = model(X_b).cpu().numpy()
        # Enforce monotonicity per-sample
        preds = np.sort(preds, axis=1)
        all_preds.append(preds)
        all_targets.append(y_b.numpy())

all_preds   = np.concatenate(all_preds, axis=0)    # (N_test, 5)
all_targets = np.concatenate(all_targets, axis=0)   # (N_test,)

print(f"Test samples: {len(all_targets)}")
print(f"Prediction shape: {all_preds.shape}")

## 5. Calibration Analysis

For each quantile τ, compute the fraction of test samples where the actual salary falls **below** the predicted τ-quantile. Ideal calibration: the fraction equals τ.

In [ ]:
calibration = {}
print(f"{'Quantile':>10s}  {'Nominal':>8s}  {'Actual':>8s}  {'Δ':>6s}")
print("-" * 40)

for i, q in enumerate(QUANTILES):
    frac_below = (all_targets < all_preds[:, i]).mean()
    delta = frac_below - q
    calibration[q] = frac_below
    status = "✅" if abs(delta) <= 0.05 else "⚠️"
    print(f"  q{int(q*100):>2d}       {q:.2f}      {frac_below:.3f}   {delta:+.3f}  {status}")

# Acceptance criterion: within ±5 percentage points
max_delta = max(abs(v - k) for k, v in calibration.items())
print(f"\nMax calibration error: {max_delta:.3f}")
print("PASS" if max_delta <= 0.05 else "FAIL (> ±5 pp)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

nominal = list(calibration.keys())
actual  = list(calibration.values())

# Perfect calibration line
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')

# ±5pp band
ax.fill_between([0, 1], [-.05, .95], [.05, 1.05], alpha=0.1, color='green',
                label='±5 pp acceptance band')

# Actual calibration
ax.scatter(nominal, actual, s=80, zorder=5, color='#e74c3c', edgecolors='white',
           linewidth=1.5)
ax.plot(nominal, actual, color='#e74c3c', linewidth=2, label='Model calibration')

for q, a in zip(nominal, actual):
    ax.annotate(f'q{int(q*100)}', (q, a), textcoords='offset points',
                xytext=(8, 8), fontsize=9)

ax.set_xlabel('Nominal quantile (τ)')
ax.set_ylabel('Actual fraction below prediction')
ax.set_title('Quantile Calibration Plot')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.set_aspect('equal')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Median Prediction MAE

**Acceptance criterion:** MAE < $15,000 on the test set.

In [ ]:
median_idx = list(QUANTILES).index(0.50)
median_preds = all_preds[:, median_idx]

residuals = all_targets - median_preds
mae = np.abs(residuals).mean()
rmse = np.sqrt((residuals ** 2).mean())
mape = (np.abs(residuals) / np.clip(all_targets, 1, None)).mean() * 100

print(f"Median prediction (q50) metrics:")
print(f"  MAE:  ${mae:,.0f}")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  MAPE: {mape:.1f}%")
print()
print("PASS" if mae < 15_000 else "FAIL (MAE ≥ $15K)")

## 7. Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# (a) Residual histogram
ax = axes[0]
ax.hist(residuals, bins=50, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Residual (actual − predicted q50)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# (b) Predicted vs. Actual scatter
ax = axes[1]
ax.scatter(median_preds, all_targets, s=8, alpha=0.4, color='#2ecc71')
lims = [min(median_preds.min(), all_targets.min()),
        max(median_preds.max(), all_targets.max())]
ax.plot(lims, lims, 'r--', linewidth=1, label='y = x')
ax.set_xlabel('Predicted salary (q50)')
ax.set_ylabel('Actual salary')
ax.set_title('Predicted vs. Actual')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend()

# (c) Residual vs. Predicted (homoscedasticity check)
ax = axes[2]
ax.scatter(median_preds, residuals, s=8, alpha=0.4, color='#9b59b6')
ax.axhline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Predicted salary (q50)')
ax.set_ylabel('Residual')
ax.set_title('Residuals vs. Predicted')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 8. Prediction Interval Fan Chart

Visualise the predicted quantile intervals for a sorted subset of test samples.

In [ ]:
# Sort by predicted median for a clean fan chart
sort_idx = np.argsort(median_preds)
N_show = min(200, len(sort_idx))  # show a subset if large
step = max(1, len(sort_idx) // N_show)
sel = sort_idx[::step][:N_show]

xs = np.arange(len(sel))

fig, ax = plt.subplots(figsize=(12, 5))

# Bands: q10–q90, q25–q75
ax.fill_between(xs, all_preds[sel, 0], all_preds[sel, 4],
                alpha=0.15, color='#3498db', label='q10–q90')
ax.fill_between(xs, all_preds[sel, 1], all_preds[sel, 3],
                alpha=0.3, color='#3498db', label='q25–q75')
ax.plot(xs, all_preds[sel, 2], color='#2c3e50', linewidth=1.5, label='q50 (median)')
ax.scatter(xs, all_targets[sel], s=10, color='#e74c3c', alpha=0.6, label='Actual', zorder=5)

ax.set_xlabel('Test samples (sorted by predicted median)')
ax.set_ylabel('Salary ($)')
ax.set_title('Salary Prediction Fan Chart')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Prediction Interval Coverage

In [ ]:
# Interval coverage: what fraction of actuals fall within each band?
intervals = [
    ("q10–q90 (80%)", 0, 4, 0.80),
    ("q25–q75 (50%)", 1, 3, 0.50),
]

print(f"{'Interval':>20s}  {'Expected':>9s}  {'Actual':>8s}  {'Δ':>6s}")
print("-" * 50)

for name, lo_idx, hi_idx, expected in intervals:
    in_band = ((all_targets >= all_preds[:, lo_idx]) & 
               (all_targets <= all_preds[:, hi_idx])).mean()
    delta = in_band - expected
    status = "✅" if abs(delta) <= 0.05 else "⚠️"
    print(f"{name:>20s}  {expected:>8.0%}      {in_band:.3f}   {delta:+.3f}  {status}")

## 10. Per-Quantile MAE

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

q_labels = [f"q{int(q*100)}" for q in QUANTILES]
q_maes   = [np.abs(all_targets - all_preds[:, i]).mean() for i in range(len(QUANTILES))]

bars = ax.bar(q_labels, q_maes, color=['#1abc9c', '#3498db', '#2c3e50', '#e67e22', '#e74c3c'],
              edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, q_maes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('MAE ($)')
ax.set_title('Mean Absolute Error by Quantile')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

## 11. Single-Sample Inference Demo

Demonstrate the `predict_salary()` API with a test embedding.

In [ ]:
# Pick a random test sample
sample_idx = 42
sample_emb = test_ds.X[sample_idx].numpy()[:EMB_DIM]  # strip extra features if any
actual_salary = test_ds.y[sample_idx].item()

result = predict_salary(model, sample_emb)

print(f"Actual salary: ${actual_salary:,.0f}")
print(f"Predicted quantiles:")
for k, v in result.items():
    print(f"  {k}: ${v:,.0f}")

## 12. Summary

| Metric | Target | Result |
|--------|--------|--------|
| Quantile calibration | ±5 pp for all τ | See §5 |
| Median MAE | < $15K | See §6 |
| q10–q90 coverage | ~80% | See §9 |
| q25–q75 coverage | ~50% | See §9 |

### Next steps
- Swap `USE_REAL_DATA = True` once Phase 2 embeddings are available
- Add extra features (experience level, work type) from Phase 1
- Hyperparameter tuning (hidden dims, dropout, learning rate)
- Save model checkpoint via `scripts/train_salary_model.py` for production use